In [40]:
import sys
sys.path.append('..')
from sqlalchemy import create_engine, text
import os

import geopandas as gpd
import pandas as pd
from geoalchemy2 import Geometry

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

POSTGRES_UTEA['DATABASE'] = 'utea_precision'

In [41]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

def obtener_lotes_para_segmentar():
    engine = obtener_engine()
    
    # Filtramos por lineas_creadas = True y segmentos_creados = False
    query = text("""
        SELECT * FROM siembra_surcado.data_lote 
        WHERE lineas_creadas = :lineas 
          AND segmentos_creados = :segmentos
    """)
    
    try:
        with engine.connect() as conn:
            df = pd.read_sql(
                query, 
                conn, 
                params={
                    "lineas": True, 
                    "segmentos": False
                }
            )
        return df
    except Exception as e:
        print(f"Error al obtener lotes para segmentar: {e}")
        return None

In [42]:
# Uso
lotes_a_segmentar = obtener_lotes_para_segmentar()

In [43]:
lotes_a_segmentar

,id,unidad_01,unidad_02,unidad_05,campanha,fecha_de_registro,puntos_cargados,lineas_creadas,segmentos_creados,desviacion_calculada,velocidad_calculada
0,5,2238,EL PARAISO,L7.1,2026,2026-03-08,True,True,False,False,False
1,6,2238,EL PARAISO,L7.2,2026,2026-03-08,True,True,False,False,False
2,7,2250,COSORIOCITO,L10.1,2026,2026-03-08,True,True,False,False,False
3,8,2250,COSORIOCITO,L10.2,2026,2026-03-08,True,True,False,False,False
4,9,2250,COSORIOCITO,L11,2026,2026-03-08,True,True,False,False,False
5,10,2250,COSORIOCITO,L12,2026,2026-03-08,True,True,False,False,False
6,11,2250,COSORIOCITO,L7,2026,2026-03-08,True,True,False,False,False
7,12,259,SANTA ANA DE PAILON,L18,2026,2026-03-08,True,True,False,False,False
8,13,30,CAMPO DULCE,ER-L1.2,2026,2026-03-08,True,True,False,False,False
9,14,2308,CAMBERRA--AGUILERA,C13,2026,2026-03-08,True,True,False,False,False


In [44]:
def procesar_segmentacion_exacta():
    engine = obtener_engine()
    
    # 1. Recorrer los IDs de data_lote (lineas_creadas=True, segmentos_creados=False)
    query_lotes = text("""
        SELECT id FROM siembra_surcado.data_lote 
        WHERE lineas_creadas = True AND segmentos_creados = False
    """)
    
    try:
        with engine.begin() as conn:
            lotes = conn.execute(query_lotes).fetchall()
            
            for (lote_id,) in lotes:
                print(f"--- Procesando Lote ID: {lote_id} ---")

                # 2. Seleccionar la(s) linea(s) de lineas_completas para este ID
                query_lineas = text("""
                    SELECT id, geom FROM siembra_surcado.lineas_completas 
                    WHERE data_lote_id = :lote_id
                """)
                lineas = conn.execute(query_lineas, {"lote_id": lote_id}).fetchall()

                for linea_id, linea_geom in lineas:
                    # 3. Seleccionar puntos por INTERSECCIÓN con la línea y filtrar por lote_id
                    # Ordenamos por isotime para garantizar la secuencia correcta
                    query_puntos = text("""
                        SELECT heading, distance, swathwidth, tilltype, sectionid, 
                               elevation, geom, isotime
                        FROM siembra_surcado.detalles_lote_utm
                        WHERE lote_id = :lote_id 
                          AND ST_Intersects(geom, :linea_geom)
                        ORDER BY isotime ASC
                    """)
                    puntos = conn.execute(query_puntos, {
                        "lote_id": lote_id, 
                        "linea_geom": linea_geom
                    }).fetchall()

                    # 4. Recorrer los puntos para crear segmentos punto a punto
                    for i in range(len(puntos) - 1):
                        punto_actual = puntos[i]
                        punto_siguiente = puntos[i+1] # El segmento hereda datos de este

                        # Crear el segmento (LineString) entre punto i y punto i+1
                        # Usamos los datos del punto_siguiente (segundo punto)
                        insert_segmento = text("""
                            INSERT INTO siembra_surcado.segmentos_lineas (
                                data_lote_id, heading, distance, swathwidth, 
                                tilltype, sectionid, elevation, isotime, geom
                            ) VALUES (
                                :lote_id, :h, :d, :sw, :tt, :sid, :el, :iso, 
                                ST_MakeLine(:geom_a, :geom_b)
                            )
                        """)

                        conn.execute(insert_segmento, {
                            "lote_id": lote_id,
                            "h": punto_siguiente.heading,
                            "d": punto_siguiente.distance,
                            "sw": punto_siguiente.swathwidth,
                            "tt": punto_siguiente.tilltype,
                            "sid": punto_siguiente.sectionid,
                            "el": punto_siguiente.elevation,
                            "iso": punto_siguiente.isotime,
                            "geom_a": punto_actual.geom,
                            "geom_b": punto_siguiente.geom
                        })

                # 5. Marcar lote como segmentado al terminar todas sus líneas
                conn.execute(text("""
                    UPDATE siembra_surcado.data_lote 
                    SET segmentos_creados = True 
                    WHERE id = :lote_id
                """), {"lote_id": lote_id})
                
                print(f"Lote {lote_id} segmentado por intersección exitosamente.")

    except Exception as e:
        print(f"Error en el proceso: {e}")


In [45]:
# Ejecutar proceso
procesar_segmentacion_exacta()

--- Procesando Lote ID: 5 ---
Lote 5 segmentado por intersección exitosamente.
--- Procesando Lote ID: 6 ---
Lote 6 segmentado por intersección exitosamente.
--- Procesando Lote ID: 7 ---
Lote 7 segmentado por intersección exitosamente.
--- Procesando Lote ID: 8 ---
Lote 8 segmentado por intersección exitosamente.
--- Procesando Lote ID: 9 ---
Lote 9 segmentado por intersección exitosamente.
--- Procesando Lote ID: 10 ---
Lote 10 segmentado por intersección exitosamente.
--- Procesando Lote ID: 11 ---
Lote 11 segmentado por intersección exitosamente.
--- Procesando Lote ID: 12 ---
Lote 12 segmentado por intersección exitosamente.
--- Procesando Lote ID: 13 ---
Lote 13 segmentado por intersección exitosamente.
--- Procesando Lote ID: 14 ---
Lote 14 segmentado por intersección exitosamente.
--- Procesando Lote ID: 15 ---
Lote 15 segmentado por intersección exitosamente.
--- Procesando Lote ID: 16 ---
Lote 16 segmentado por intersección exitosamente.
--- Procesando Lote ID: 1 ---
Lote 1 s

In [46]:
def calcular_desviacion_segmentos():
    engine = obtener_engine()
    
    # 1. Obtener los IDs de los lotes pendientes
    query_lotes = text("""
        SELECT id FROM siembra_surcado.data_lote 
        WHERE desviacion_calculada = False and segmentos_creados = True
    """)
    
    try:
        with engine.begin() as conn:
            lotes = conn.execute(query_lotes).fetchall()
            
            for (lote_id,) in lotes:
                print(f"--- Procesando Lote ID: {lote_id} ---")

                # Esta consulta usa RANK para identificar el vecino más cercano en tiempo
                # priorizando los menores (pasadas previas) y luego los mayores (pasadas posteriores)
                query_proceso = text("""
                    WITH calculo_base AS (
                        SELECT 
                            s1.id as seg_id,
                            s1.isotime as iso_orig,
                            ST_LineInterpolatePoint(s1.geom, 0.5) as centro,
                            ST_Azimuth(ST_StartPoint(s1.geom), ST_EndPoint(s1.geom)) as azimut,
                            ((s1.swathwidth + 1) * 2) as largo_sonda
                        FROM siembra_surcado.segmentos_lineas s1
                        WHERE s1.data_lote_id = :lote_id
                    ),
                    sonda_perpendicular AS (
                        SELECT 
                            seg_id,
                            iso_orig,
                            ST_MakeLine(
                                ST_Project(centro, largo_sonda/2, azimut + pi()/2),
                                ST_Project(centro, largo_sonda/2, azimut - pi()/2)
                            ) as linea_sonda,
                            centro
                        FROM calculo_base
                    ),
                    vecinos_detectados AS (
                        SELECT 
                            sp.seg_id,
                            sp.centro,
                            s2.geom as geom_vecino,
                            s2.isotime as iso_vecino,
                            -- Determinamos si es un vecino pasado (0) o futuro (1)
                            CASE WHEN s2.isotime < sp.iso_orig THEN 0 ELSE 1 END as orden_cronologico,
                            -- Calculamos la diferencia de tiempo absoluta para encontrar el más cercano
                            ABS(EXTRACT(EPOCH FROM (s2.isotime - sp.iso_orig))) as diferencia_tiempo
                        FROM sonda_perpendicular sp
                        JOIN siembra_surcado.segmentos_lineas s2 
                          ON ST_Intersects(sp.linea_sonda, s2.geom)
                        WHERE s2.data_lote_id = :lote_id
                          AND s2.id != sp.seg_id
                    ),
                    priorizacion AS (
                        SELECT 
                            seg_id,
                            centro,
                            geom_vecino,
                            ROW_NUMBER() OVER (
                                PARTITION BY seg_id 
                                ORDER BY 
                                    orden_cronologico ASC, -- Prioriza los pasados (0) sobre futuros (1)
                                    diferencia_tiempo ASC   -- Prioriza el más cercano en tiempo
                            ) as prioridad
                        FROM vecinos_detectados
                    )
                    UPDATE siembra_surcado.segmentos_lineas target
                    SET desviacion = ST_Distance(p.centro, p.geom_vecino)
                    FROM priorizacion p
                    WHERE target.id = p.seg_id 
                      AND p.prioridad = 1;
                """)
                
                conn.execute(query_proceso, {"lote_id": lote_id})

                # 3. Calcular Variación y Categoría manejando los casos sin vecino
                # Usamos COALESCE(desviacion, swathwidth) para que si es NULL, use el ancho teórico
                query_categorizacion = text("""
                    UPDATE siembra_surcado.segmentos_lineas
                    SET 
                        variacion = ABS(COALESCE(desviacion, swathwidth) - swathwidth),
                        categoria_variacion = CASE 
                            WHEN ABS(COALESCE(desviacion, swathwidth) - swathwidth) <= 0.05 THEN '1) <5 - OPTIMO'
                            WHEN ABS(COALESCE(desviacion, swathwidth) - swathwidth) <= 0.10 THEN '2) 5 a 10 - ACEPTABLE'
                            WHEN ABS(COALESCE(desviacion, swathwidth) - swathwidth) <= 0.15 THEN '3) 10 a 15 - RIESGO MODERADO'
                            ELSE '4) >15 - RIESGO ALTO'
                        END
                    WHERE data_lote_id = :lote_id;
                """)
                conn.execute(query_categorizacion, {"lote_id": lote_id})
                
                # 4. Actualización del estado del lote
                conn.execute(text("""
                    UPDATE siembra_surcado.data_lote 
                    SET desviacion_calculada = True
                    WHERE id = :lote_id
                """), {"lote_id": lote_id})
                print(f"  > Lote {lote_id} procesado íntegramente.")
                
    except Exception as e:
        print(f"Error en el proceso: {e}")

In [47]:
calcular_desviacion_segmentos()

--- Procesando Lote ID: 5 ---
  > Lote 5 procesado íntegramente.
--- Procesando Lote ID: 6 ---
  > Lote 6 procesado íntegramente.
--- Procesando Lote ID: 7 ---
  > Lote 7 procesado íntegramente.
--- Procesando Lote ID: 8 ---
  > Lote 8 procesado íntegramente.
--- Procesando Lote ID: 9 ---
  > Lote 9 procesado íntegramente.
--- Procesando Lote ID: 10 ---
  > Lote 10 procesado íntegramente.
--- Procesando Lote ID: 11 ---
  > Lote 11 procesado íntegramente.
--- Procesando Lote ID: 12 ---
  > Lote 12 procesado íntegramente.
--- Procesando Lote ID: 13 ---
  > Lote 13 procesado íntegramente.
--- Procesando Lote ID: 14 ---
  > Lote 14 procesado íntegramente.
--- Procesando Lote ID: 15 ---
  > Lote 15 procesado íntegramente.
--- Procesando Lote ID: 16 ---
  > Lote 16 procesado íntegramente.
--- Procesando Lote ID: 1 ---
  > Lote 1 procesado íntegramente.
--- Procesando Lote ID: 18 ---
  > Lote 18 procesado íntegramente.
--- Procesando Lote ID: 19 ---
  > Lote 19 procesado íntegramente.
--- Pro

In [48]:
def calcular_velocidad_segmentos():
    engine = obtener_engine()
    
    # 1. Obtener los IDs de los lotes pendientes
    query_lotes = text("""
        SELECT id FROM siembra_surcado.data_lote 
        WHERE velocidad_calculada = False AND segmentos_creados = True
    """)
    
    try:
        with engine.begin() as conn:
            lotes = conn.execute(query_lotes).fetchall()
            
            for (lote_id,) in lotes:
                print(f"--- Calculando Velocidades y Categorías Lote ID: {lote_id} ---")

                # 2. SQL con lógica de categorización añadida
                query_proceso_velocidad = text("""
                    WITH calculo_secuencial AS (
                        SELECT 
                            id,
                            isotime,
                            LAG(isotime) OVER (ORDER BY isotime) as iso_previo,
                            distance
                        FROM siembra_surcado.segmentos_lineas
                        WHERE data_lote_id = :lote_id
                    ),
                    metricas_calculadas AS (
                        SELECT 
                            id,
                            iso_previo,
                            CASE 
                                WHEN iso_previo IS NOT NULL AND (EXTRACT(EPOCH FROM (isotime - iso_previo))) > 0
                                THEN (distance / EXTRACT(EPOCH FROM (isotime - iso_previo))) * 3.6
                                ELSE 0 
                            END as velocidad_kmh
                        FROM calculo_secuencial
                    ),
                    categorizacion AS (
                        SELECT 
                            id,
                            iso_previo,
                            velocidad_kmh,
                            CASE 
                                WHEN velocidad_kmh <= 3.8 THEN '1) 3.3 km/h'
                                WHEN velocidad_kmh <= 4.8 THEN '2) 4.3 km/h'
                                WHEN velocidad_kmh <= 5.8 THEN '3) 5.3 km/h'
                                WHEN velocidad_kmh <= 6.8 THEN '4) 6.3 km/h'
                                WHEN velocidad_kmh <= 7.8 THEN '5) 7.3 km/h'
                                ELSE '6) >7.3 km/h'
                            END as cat_vel
                        FROM metricas_calculadas
                    )
                    UPDATE siembra_surcado.segmentos_lineas target
                    SET 
                        isotime_anterior = c.iso_previo,
                        velocidad = c.velocidad_kmh,
                        categoria_velocidad = c.cat_vel
                    FROM categorizacion c
                    WHERE target.id = c.id;
                """)
                
                conn.execute(query_proceso_velocidad, {"lote_id": lote_id})

                # 3. Marcar el lote como procesado
                conn.execute(text("""
                    UPDATE siembra_surcado.data_lote 
                    SET velocidad_calculada = True
                    WHERE id = :lote_id
                """), {"lote_id": lote_id})
                
                print(f"  > Velocidades y Categorías del Lote {lote_id} actualizadas.")

    except Exception as e:
        print(f"Error en el cálculo de velocidad: {e}")

In [49]:
calcular_velocidad_segmentos()

--- Calculando Velocidades y Categorías Lote ID: 5 ---
  > Velocidades y Categorías del Lote 5 actualizadas.
--- Calculando Velocidades y Categorías Lote ID: 6 ---
  > Velocidades y Categorías del Lote 6 actualizadas.
--- Calculando Velocidades y Categorías Lote ID: 7 ---
  > Velocidades y Categorías del Lote 7 actualizadas.
--- Calculando Velocidades y Categorías Lote ID: 8 ---
  > Velocidades y Categorías del Lote 8 actualizadas.
--- Calculando Velocidades y Categorías Lote ID: 9 ---
  > Velocidades y Categorías del Lote 9 actualizadas.
--- Calculando Velocidades y Categorías Lote ID: 10 ---
  > Velocidades y Categorías del Lote 10 actualizadas.
--- Calculando Velocidades y Categorías Lote ID: 11 ---
  > Velocidades y Categorías del Lote 11 actualizadas.
--- Calculando Velocidades y Categorías Lote ID: 12 ---
  > Velocidades y Categorías del Lote 12 actualizadas.
--- Calculando Velocidades y Categorías Lote ID: 13 ---
  > Velocidades y Categorías del Lote 13 actualizadas.
--- Calcula